# Resume Fraud Detection Module
*Isolation Forest + Rule-based Hard Flags*

In [ ]:
! pip install pdfplumber scikit-learn python-docx -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 89.1 MB/s eta 0:00:00


In [ ]:
import pdfplumber
from docx import Document
import re, ast
from collections import Counter
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

def extract_text_from_pdf(file_path):
    text = ""
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                text += t + "\n"
    return text

def extract_text_from_docx(file_path):
    doc = Document(file_path)
    return "\n".join(p.text for p in doc.paragraphs)

def parse_resume(file_path):
    if file_path.endswith(".pdf"):   return extract_text_from_pdf(file_path)
    elif file_path.endswith(".docx"): return extract_text_from_docx(file_path)
    else: raise ValueError("Unsupported format. Use .pdf or .docx")

## Configuration

In [ ]:
job_description = """
We are looking for a Machine Learning Engineer or NLP Engineer with strong skills in Python,
Natural Language Processing, BERT, Deep Learning, and data preprocessing.
Experience with spaCy, Scikit-learn, and building AI models is preferred.
"""

skills_list = [
    "python", "machine learning", "deep learning", "nlp", "bert",
    "spacy", "scikit-learn", "data analysis", "statistics",
    "tensorflow", "pytorch", "sql", "mysql", "hadoop"
]

def extract_skills(text):
    return [s for s in skills_list if re.search(r'\b' + s + r'\b', text.lower())]

jd_skills = extract_skills(job_description)

## Feature Functions

> **Critical design rule:** All features must be *comparable* between the CSV training data and real PDF resumes. Features removed from earlier versions:
> - `skill_match` / `skill_gap` — depend on which JD is used; wildly different between training data and real resumes
> - `length` — training rows are 60-word snippets; real PDFs are 400-800 words; same number means completely different things
> - `exp_gap` — zero variance in training data, causes StandardScaler to break

In [ ]:
def repetition_score(skills):
    """Fraction of skill list entries that are duplicates."""
    if not skills: return 0
    return 1 - (len(set(skills)) / (len(skills) + 1))

def word_repetition_score(text):
    """How much non-stopword text is repeated. High = suspicious."""
    words = text.lower().split()
    if not words: return 0
    counts = Counter(words)
    stop = {"i","am","a","the","in","and","for","with","of","to","as","is",
            "my","have","has","at","on","an","by","be","are","was","were"}
    repeated = sum(c - 1 for w, c in counts.items() if c > 1 and w not in stop)
    return repeated / (len(words) + 1)

def consecutive_word_repeat_score(text):
    """
    Detects 'Python Python Python' padding.
    Normal resume: 1. Suspicious: 2. Fraud: 3+.
    This is the strongest fraud signal for padded resumes.
    """
    words = [re.sub(r'[^a-z]', '', w) for w in text.lower().split()]
    max_run, cur = 1, 1
    for i in range(1, len(words)):
        if words[i] and words[i] == words[i - 1]:
            cur += 1
            max_run = max(max_run, cur)
        else:
            cur = 1
    return max_run

def buzzword_score(text):
    """Density of self-aggrandizing language."""
    buzzwords = [
        "expert", "advanced", "proficient", "specialist", "experienced",
        "highly", "everything", "all technologies", "best", "master",
        "guru", "ninja", "rockstar", "wizard", "exceptional"
    ]
    count = sum(text.lower().count(w) for w in buzzwords)
    return count / (len(text.split()) + 1)

def content_richness(text):
    """Vocabulary diversity. Low = repetitive/padded content."""
    words = text.split()
    return len(set(words)) / len(words) if words else 0

def has_projects_section(text):
    """Returns 0 if projects section exists but is empty or says 'None'."""
    tl = text.lower()
    if "projects" in tl:
        idx = tl.find("projects")
        after = tl[idx:idx + 60]
        if "none" in after or after.strip().endswith("projects"):
            return 0
    return 1

def experience_credibility_gap(text):
    """
    Flags impossible timelines. 'B.Tech 2023' + '15 years exp' = fraud.
    Returns 0 (plausible), 1 (suspicious), 2+ (impossible).
    NOTE: Used only in hard_flag_check, NOT as a model feature
    (zero variance in training data would break StandardScaler).
    """
    exp = re.search(r'(\d+)\s*years?', text.lower())
    grad = re.search(r'(202\d|201\d)', text)
    if exp and grad:
        ey, gy = int(exp.group(1)), int(grad.group(1))
        if gy >= 2018 and ey >= 5:
            return min(ey / 5.0, 3.0)
    return 0.0

def is_student_resume(text):
    """Detects genuine student/fresher profiles to avoid false positives."""
    indicators = ["student", "intern", "fresher", "cgpa", "pursuing", "present",
                  "ongoing", "b.tech", "btech", "engineering student",
                  "third-year", "second-year", "first-year"]
    tl = text.lower()
    return any(w in tl for w in indicators)

## Unified Feature Extractor

This function is used for **both training (CSV rows) and inference (PDF files)**.

In [ ]:
def extract_features(raw_text, skills):
    """
    6 features — all comparable between short CSV snippets and full PDF resumes.
    Uses raw text (NOT cleaned) to preserve punctuation/spacing signals.
    """
    return [
        repetition_score(skills),                    # 0: duplicate skills ratio
        word_repetition_score(raw_text),             # 1: word repetition in prose
        consecutive_word_repeat_score(raw_text),     # 2: 'X X X' padding (KEY signal)
        buzzword_score(raw_text),                    # 3: buzzword density
        content_richness(raw_text),                  # 4: vocabulary diversity
        has_projects_section(raw_text),              # 5: projects present and non-empty
    ]

def safe_eval(x):
    try: return ast.literal_eval(x) if isinstance(x, str) else []
    except: return []

def extract_features_from_dataset(row):
    """Extracts features from a CSV training data row."""
    skills = safe_eval(row.get('skills', '[]'))
    if isinstance(skills, str):
        skills = [s.strip() for s in skills.split(',')]
    text_parts = [
        str(row.get('career_objective', '')),
        str(row.get('responsibilities', '')),
        str(row.get('responsibilities.1', '')),
    ]
    text = ' '.join(p for p in text_parts if p and p != 'nan')
    return extract_features(text, skills)

## Train the Model

In [ ]:
df = pd.read_csv("resume_data.csv")
df_clean = df[df['matched_score'] > 0.5]
print(f"Training on {len(df_clean)} high-quality resumes...")

X = np.array(df_clean.apply(extract_features_from_dataset, axis=1).tolist())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iso = IsolationForest(n_estimators=300, contamination=0.08, random_state=42)
iso.fit(X_scaled)
print("✅ Model trained.")

Training on 7736 high-quality resumes...
✅ Model trained.


## Detection Logic

**Two-layer architecture:**
1. **Hard flags** (rules) — deterministic checks for known fraud patterns. Take priority over model.
2. **Isolation Forest** — statistical anomaly detection for patterns rules don't cover.

A resume with 2+ hard flags is always 🚨 High Risk, regardless of model score.

In [ ]:
def hard_flag_check(raw_text, features):
    """Rule-based checks. These run BEFORE the model and take priority."""
    flags = []
    consec = features[2]
    buzz   = features[3]
    proj   = features[5]
    wrep   = features[1]
    exp_gap = experience_credibility_gap(raw_text)

    if consec >= 3:
        flags.append(f"Word repeated {int(consec)}x in a row — likely skill padding")
    if exp_gap >= 2.0:
        flags.append("Claims 10+ years experience but graduated recently — timeline impossible")
    elif exp_gap >= 1.0:
        flags.append("Experience years inconsistent with graduation year")
    if buzz > 0.08:
        flags.append("Excessive self-praise language (highly/expert/best overused)")
    if proj == 0:
        flags.append("Projects section is empty or says 'None'")
    if wrep > 0.25:
        flags.append("High word repetition in free text")
    return flags

def interpret_result(pred, score, hard_flags, raw_text):
    n = len(hard_flags)
    if n >= 2:
        return "🚨 High Risk Resume"
    if n == 1 and pred == -1:
        return "⚠️ Suspicious Resume"
    if pred == 1:
        return "✅ Normal Resume"
    # Model says anomaly, but no hard flags — check if it's a genuine student resume
    # (Student resumes are short and skills-light, which makes them statistically
    #  unusual vs the training data of full professional profiles)
    if is_student_resume(raw_text) and n == 0:
        return "✅ Normal Resume"
    if score < -0.19:   return "🚨 High Risk Resume"
    if score < -0.165:  return "⚠️ Suspicious Resume"
    if score < -0.13:   return "🟡 Slight Anomaly"
    return "✅ Normal Resume"

def explain(features, hard_flags, score):
    reasons = list(hard_flags)
    rep, word_rep, consec, buzz, rich, proj = features
    if rep > 0.3 and not any("padding" in f for f in reasons):
        reasons.append("Skill list contains many duplicate entries")
    if rich < 0.5:
        reasons.append("Very low vocabulary diversity")
    if not reasons:
        reasons.append("Resume looks normal" if score > -0.13
                       else "Resume deviates from normal patterns")
    return reasons

## Batch Test Function

In [ ]:
def test_resumes(resume_files, jd_text):
    """Pass a list of file paths. Prints results and returns structured output."""
    results = []
    for i, file_path in enumerate(resume_files, 1):
        print("\n" + "="*55)
        print(f"📄 Resume {i}: {file_path.split('/')[-1]}")
        print("="*55)

        raw      = parse_resume(file_path)
        skills   = extract_skills(raw)
        features = extract_features(raw, skills)

        fs    = scaler.transform([features])
        pred  = iso.predict(fs)[0]
        score = iso.decision_function(fs)[0]

        flags  = hard_flag_check(raw, features)
        result = interpret_result(pred, score, flags, raw)
        reasons = explain(features, flags, score)

        print(f"Fraud Score : {score:.4f}")
        print(f"Prediction  : {result}")
        print("Reasons:")
        for r in reasons:
            print(f"  ⚠  {r}")

        results.append({
            "resume": file_path.split('/')[-1],
            "score": round(float(score), 4),
            "prediction": result,
            "reasons": reasons
        })
    return results

In [ ]:
files = [
    "normal_resume.pdf",
    "weak_resume.pdf",
    "sus_resume.pdf",
    "high_fraud_resume.pdf",
    # Add your own resumes below:
    # "resume.pdf",
    "resume.pdf",
    "resume2.pdf",
    "resume3.pdf"
]

results = test_resumes(files, job_description)


📄 Resume 1: normal_resume.pdf
Fraud Score : -0.1078
Prediction  : ✅ Normal Resume
Reasons:
  ⚠  Resume looks normal

📄 Resume 2: weak_resume.pdf
Fraud Score : -0.1606
Prediction  : ⚠️ Suspicious Resume
Reasons:
  ⚠  Projects section is empty or says 'None'
  ⚠  Skill list contains many duplicate entries

📄 Resume 3: sus_resume.pdf
Fraud Score : -0.1861
Prediction  : 🚨 High Risk Resume
Reasons:
  ⚠  Word repeated 3x in a row — likely skill padding
  ⚠  Experience years inconsistent with graduation year
  ⚠  Excessive self-praise language (highly/expert/best overused)
  ⚠  Projects section is empty or says 'None'

📄 Resume 4: high_fraud_resume.pdf
Fraud Score : -0.1383
Prediction  : 🚨 High Risk Resume
Reasons:
  ⚠  Word repeated 6x in a row — likely skill padding
  ⚠  Claims 10+ years experience but graduated recently — timeline impossible
  ⚠  Excessive self-praise language (highly/expert/best overused)
  ⚠  Projects section is empty or says 'None'
  ⚠  High word repetition in free tex

## Results Summary Table

In [ ]:
summary = pd.DataFrame([{
    "Resume": r["resume"],
    "Score": r["score"],
    "Prediction": r["prediction"],
    "Top Reason": r["reasons"][0] if r["reasons"] else "—"
} for r in results])

summary

,Resume,Score,Prediction,Top Reason
0,normal_resume.pdf,-0.1078,✅ Normal Resume,Resume looks normal
1,weak_resume.pdf,-0.1606,⚠️ Suspicious Resume,Projects section is empty or says 'None'
2,sus_resume.pdf,-0.1861,🚨 High Risk Resume,Word repeated 3x in a row — likely skill padding
3,high_fraud_resume.pdf,-0.1383,🚨 High Risk Resume,Word repeated 6x in a row — likely skill padding
4,resume.pdf,-0.0278,✅ Normal Resume,Resume looks normal
5,resume2.pdf,-0.0754,✅ Normal Resume,Resume looks normal
6,resume3.pdf,-0.1062,✅ Normal Resume,Resume looks normal


# Skill Gap Analysis + Learning Recommendations - ATS Pipeline Module 2

In [ ]:
! pip install pdfplumber scikit-learn python-docx -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 70.6 MB/s eta 0:00:00


In [ ]:
import re
import pdfplumber
from docx import Document

def extract_text_from_pdf(file_path):
    text = ""
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                text += t + "\n"
    return text

def extract_text_from_docx(file_path):
    doc = Document(file_path)
    return "\n".join(p.text for p in doc.paragraphs)

def parse_resume(file_path):
    if file_path.endswith(".pdf"):    return extract_text_from_pdf(file_path)
    elif file_path.endswith(".docx"): return extract_text_from_docx(file_path)
    else: raise ValueError("Use .pdf or .docx")

## Skill Taxonomy Covers 50+ skills across Programming, ML/AI, Web Dev, Databases, DevOps, Cloud, and Data. Add any skill by following the same format.

In [ ]:
SKILL_TAXONOMY = {
    # Programming Languages
    "python":          {"category": "Programming", "level": "core",     "resources": [("Python for Everybody - Coursera", "https://coursera.org/specializations/python"), ("Automate the Boring Stuff (free)", "https://automatetheboringstuff.com")]},
    "java":            {"category": "Programming", "level": "core",     "resources": [("Java Programming - MOOC.fi (free)", "https://java-programming.mooc.fi"), ("Java Masterclass - Udemy", "https://udemy.com/course/java-the-complete-java-developer-course")]},
    "javascript":      {"category": "Programming", "level": "core",     "resources": [("The Odin Project (free)", "https://theodinproject.com"), ("javascript.info (free)", "https://javascript.info")]},
    "typescript":      {"category": "Programming", "level": "core",     "resources": [("TypeScript Handbook (free)", "https://typescriptlang.org/docs/handbook/intro.html"), ("Total TypeScript (free tier)", "https://totaltypescript.com")]},
    "c++":             {"category": "Programming", "level": "core",     "resources": [("learncpp.com (free)", "https://learncpp.com"), ("C++ Primer (book)", "https://www.amazon.com/dp/0321714113")]},
    "go":              {"category": "Programming", "level": "core",     "resources": [("Tour of Go (free)", "https://go.dev/tour"), ("Go by Example (free)", "https://gobyexample.com")]},
    "rust":            {"category": "Programming", "level": "advanced", "resources": [("The Rust Book (free)", "https://doc.rust-lang.org/book"), ("Rustlings (free)", "https://github.com/rust-lang/rustlings")]},
    # ML / AI
    "machine learning": {"category": "ML/AI", "level": "core",     "resources": [("ML Specialization - Coursera (Andrew Ng)", "https://coursera.org/specializations/machine-learning-introduction"), ("fast.ai Practical ML (free)", "https://course.fast.ai")]},
    "deep learning":    {"category": "ML/AI", "level": "advanced",  "resources": [("Deep Learning Specialization - Coursera", "https://coursera.org/specializations/deep-learning"), ("MIT 6.S191 (free)", "https://introtodeeplearning.com")]},
    "nlp":              {"category": "ML/AI", "level": "advanced",  "resources": [("HuggingFace NLP Course (free)", "https://huggingface.co/learn/nlp-course"), ("Stanford CS224N (free)", "https://web.stanford.edu/class/cs224n")]},
    "bert":             {"category": "ML/AI", "level": "advanced",  "resources": [("HuggingFace Transformers Docs (free)", "https://huggingface.co/docs/transformers"), ("Illustrated BERT - Jay Alammar (free)", "https://jalammar.github.io/illustrated-bert")]},
    "tensorflow":       {"category": "ML/AI", "level": "advanced",  "resources": [("TensorFlow Tutorials (free)", "https://tensorflow.org/tutorials"), ("Deep Learning with Python (book)", "https://www.manning.com/books/deep-learning-with-python")]},
    "pytorch":          {"category": "ML/AI", "level": "advanced",  "resources": [("PyTorch Tutorials (free)", "https://pytorch.org/tutorials"), ("fast.ai (uses PyTorch, free)", "https://course.fast.ai")]},
    "scikit-learn":     {"category": "ML/AI", "level": "core",      "resources": [("Scikit-learn User Guide (free)", "https://scikit-learn.org/stable/user_guide.html"), ("Hands-on ML with Scikit-Learn (book)", "https://oreilly.com/library/view/hands-on-machine-learning/9781492032632")]},
    "langchain":        {"category": "ML/AI", "level": "advanced",  "resources": [("LangChain Docs (free)", "https://python.langchain.com"), ("LangChain Crash Course - YouTube", "https://youtube.com/watch?v=lG7Uxts9SXs")]},
    "huggingface":      {"category": "ML/AI", "level": "advanced",  "resources": [("HuggingFace Course (free)", "https://huggingface.co/learn/nlp-course"), ("HuggingFace Docs (free)", "https://huggingface.co/docs")]},
    "spacy":            {"category": "ML/AI", "level": "core",      "resources": [("spaCy Course (free)", "https://course.spacy.io"), ("spaCy Docs (free)", "https://spacy.io/usage")]},
    "opencv":           {"category": "ML/AI", "level": "core",      "resources": [("OpenCV Python Tutorials (free)", "https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html"), ("PyImageSearch Blog (free)", "https://pyimagesearch.com")]},
    "generative ai":    {"category": "ML/AI", "level": "advanced",  "resources": [("Google GenAI Course (free)", "https://ai.google.dev/gemini-api/docs"), ("Prompt Engineering Guide (free)", "https://promptingguide.ai")]},
    "statistics":       {"category": "ML/AI", "level": "core",      "resources": [("StatQuest - YouTube (free)", "https://youtube.com/@statquest"), ("Khan Academy Statistics (free)", "https://khanacademy.org/math/statistics-probability")]},
    # Web / Backend
    "react":            {"category": "Web Dev", "level": "core",    "resources": [("React Official Docs (free)", "https://react.dev/learn"), ("Full Stack Open (free)", "https://fullstackopen.com")]},
    "vue":              {"category": "Web Dev", "level": "core",    "resources": [("Vue Official Guide (free)", "https://vuejs.org/guide/introduction.html"), ("Vue Mastery (free tier)", "https://vuemastery.com")]},
    "node.js":          {"category": "Web Dev", "level": "core",    "resources": [("Node.js Learn (free)", "https://nodejs.org/en/learn"), ("The Odin Project (free)", "https://theodinproject.com")]},
    "fastapi":          {"category": "Web Dev", "level": "core",    "resources": [("FastAPI Tutorial (free)", "https://fastapi.tiangolo.com/tutorial"), ("FastAPI Full Course - YouTube", "https://youtube.com/watch?v=7t2alSnE2-I")]},
    "flask":            {"category": "Web Dev", "level": "core",    "resources": [("Flask Mega-Tutorial (free)", "https://blog.miguelgrinberg.com/post/the-flask-mega-tutorial-part-i-hello-world"), ("Flask Docs (free)", "https://flask.palletsprojects.com")]},
    "django":           {"category": "Web Dev", "level": "core",    "resources": [("Django Official Tutorial (free)", "https://docs.djangoproject.com/en/5.0/intro/tutorial01"), ("Django for Beginners (book)", "https://djangoforbeginners.com")]},
    # Databases
    "sql":              {"category": "Databases", "level": "core",    "resources": [("SQLZoo (free)", "https://sqlzoo.net"), ("Mode SQL Tutorial (free)", "https://mode.com/sql-tutorial")]},
    "mysql":            {"category": "Databases", "level": "core",    "resources": [("MySQL Tutorial (free)", "https://mysqltutorial.org"), ("MySQL Docs (free)", "https://dev.mysql.com/doc")]},
    "mongodb":          {"category": "Databases", "level": "core",    "resources": [("MongoDB University (free)", "https://learn.mongodb.com"), ("MongoDB Docs (free)", "https://www.mongodb.com/docs/manual")]},
    "postgresql":       {"category": "Databases", "level": "core",    "resources": [("PostgreSQL Tutorial (free)", "https://postgresqltutorial.com"), ("PostgreSQL Docs (free)", "https://postgresql.org/docs")]},
    "redis":            {"category": "Databases", "level": "core",    "resources": [("Redis University (free)", "https://university.redis.com"), ("Redis Docs (free)", "https://redis.io/docs")]},
    # DevOps / Cloud
    "docker":           {"category": "DevOps", "level": "core",      "resources": [("Docker Getting Started (free)", "https://docs.docker.com/get-started"), ("TechWorld with Nana - Docker (YouTube)", "https://youtube.com/watch?v=3c-iBn73dDE")]},
    "kubernetes":       {"category": "DevOps", "level": "advanced",  "resources": [("Kubernetes Basics (free)", "https://kubernetes.io/docs/tutorials/kubernetes-basics"), ("KodeKloud K8s Free Tier", "https://kodekloud.com/courses/kubernetes-for-the-absolute-beginners-hands-on")]},
    "aws":              {"category": "Cloud",   "level": "core",     "resources": [("AWS Skill Builder (free)", "https://skillbuilder.aws"), ("AWS Cloud Practitioner Essentials", "https://aws.amazon.com/training/learn-about/cloud-practitioner")]},
    "azure":            {"category": "Cloud",   "level": "core",     "resources": [("Microsoft Learn Azure (free)", "https://learn.microsoft.com/en-us/training/azure"), ("AZ-900 Study Guide (free)", "https://learn.microsoft.com/en-us/certifications/azure-fundamentals")]},
    "gcp":              {"category": "Cloud",   "level": "core",     "resources": [("Google Cloud Skills Boost (free)", "https://cloudskillsboost.google"), ("GCP Associate Engineer Prep", "https://cloud.google.com/learn/certification/cloud-engineer")]},
    "git":              {"category": "DevOps",  "level": "core",     "resources": [("Pro Git Book (free)", "https://git-scm.com/book/en/v2"), ("Learn Git Branching (interactive, free)", "https://learngitbranching.js.org")]},
    "linux":            {"category": "DevOps",  "level": "core",     "resources": [("Linux Journey (free)", "https://linuxjourney.com"), ("The Linux Command Line (free)", "https://linuxcommand.org/tlcl.php")]},
    # Data
    "pandas":           {"category": "Data", "level": "core",        "resources": [("Pandas Tutorials (free)", "https://pandas.pydata.org/docs/getting_started/intro_tutorials"), ("Kaggle Pandas Course (free)", "https://kaggle.com/learn/pandas")]},
    "numpy":            {"category": "Data", "level": "core",        "resources": [("NumPy Quickstart (free)", "https://numpy.org/doc/stable/user/quickstart.html"), ("NumPy for Beginners (free)", "https://numpy.org/doc/stable/user/absolute_beginners.html")]},
    "data analysis":    {"category": "Data", "level": "core",        "resources": [("freeCodeCamp Data Analysis (free)", "https://freecodecamp.org/learn/data-analysis-with-python"), ("Kaggle Learn (free)", "https://kaggle.com/learn")]},
    "hadoop":           {"category": "Data", "level": "advanced",    "resources": [("Hadoop Tutorial (free)", "https://tutorialspoint.com/hadoop"), ("Hadoop: The Definitive Guide (book)", "https://oreilly.com/library/view/hadoop-the-definitive/9781491901687")]},
    "spark":            {"category": "Data", "level": "advanced",    "resources": [("Spark Official Docs (free)", "https://spark.apache.org/docs/latest"), ("Databricks Free Training", "https://www.databricks.com/learn/training/home")]},
    "power bi":         {"category": "Data", "level": "core",        "resources": [("Microsoft Power BI Learning (free)", "https://learn.microsoft.com/en-us/training/powerplatform/power-bi"), ("Guy in a Cube - YouTube (free)", "https://youtube.com/@GuyInACube")]},
    "tableau":          {"category": "Data", "level": "core",        "resources": [("Tableau Training (free)", "https://www.tableau.com/learn/training"), ("Tableau for Beginners - YouTube", "https://youtube.com/watch?v=TPMlZxRRaBQ")]},
}

print(f"Taxonomy: {len(SKILL_TAXONOMY)} skills across",
      len(set(v['category'] for v in SKILL_TAXONOMY.values())), "categories")

Taxonomy: 45 skills across 7 categories


## Analysis Functions

In [ ]:
def extract_candidate_skills(resume_text):
    found = set()
    text_lower = resume_text.lower()
    for skill in SKILL_TAXONOMY:
        if re.search(r'\b' + re.escape(skill) + r'\b', text_lower):
            found.add(skill)
    return found

def parse_jd_skills(jd_text):
    found = set()
    jd_lower = jd_text.lower()
    for skill in SKILL_TAXONOMY:
        if re.search(r'\b' + re.escape(skill) + r'\b', jd_lower):
            found.add(skill)
    return found

def skill_gap_analysis(resume_text, jd_text, candidate_name="Candidate"):
    jd_required   = parse_jd_skills(jd_text)
    candidate_has = extract_candidate_skills(resume_text)
    matched = candidate_has & jd_required
    missing = jd_required - candidate_has
    extra   = candidate_has - jd_required
    match_pct = round(len(matched) / len(jd_required) * 100) if jd_required else 0

    recommendations = []
    for skill in missing:
        info = SKILL_TAXONOMY[skill]
        recommendations.append({
            "skill": skill,
            "category": info["category"],
            "level": info["level"],
            "resources": info["resources"],
        })
    recommendations.sort(key=lambda x: (0 if x["level"] == "core" else 1, x["category"], x["skill"]))

    return {
        "candidate":       candidate_name,
        "match_percent":   match_pct,
        "jd_required":     sorted(jd_required),
        "matched":         sorted(matched),
        "missing":         sorted(missing),
        "extra_skills":    sorted(extra),
        "recommendations": recommendations,
    }

def print_gap_report(result):
    w = 60
    print("\n" + "="*w)
    print(f"  SKILL GAP REPORT -- {result['candidate']}")
    print("="*w)
    print(f"  Match Score : {result['match_percent']}%  ({len(result['matched'])}/{len(result['jd_required'])} required skills)")
    print()
    if result['matched']:
        print(f"  MATCHED ({len(result['matched'])})")
        for s in result['matched']:
            print(f"    [+] {s.title()}")
    print()
    if result['missing']:
        print(f"  MISSING ({len(result['missing'])})")
        for s in result['missing']:
            lvl = SKILL_TAXONOMY[s]['level']
            cat = SKILL_TAXONOMY[s]['category']
            print(f"    [-] {s.title():<22} [{lvl.upper():8}] {cat}")
    print()
    if result['extra_skills']:
        print(f"  BONUS SKILLS ({len(result['extra_skills'])})")
        for s in result['extra_skills']:
            print(f"    [*] {s.title()}")
    print()
    if result['recommendations']:
        print("  LEARNING ROADMAP (in order)")
        print("  " + "-"*55)
        current_level = None
        for rec in result['recommendations']:
            if rec['level'] != current_level:
                current_level = rec['level']
                label = "CORE SKILLS (Start here)" if current_level == "core" else "ADVANCED SKILLS (After core)"
                print(f"\n  [{label}]")
            print(f"\n    {rec['skill'].title()} ({rec['category']})")
            for name, url in rec['resources']:
                print(f"      -> {name}")
                print(f"         {url}")
    print("\n" + "="*w)

## Run the Analysis

In [ ]:
# ─── CONFIG ────────────────────────────────────────────────
RESUME_FILE    = "resume2.pdf"        # change to your file
CANDIDATE_NAME = "Anushka"         # change to candidate name

JOB_DESCRIPTION = """
We are looking for a Machine Learning Engineer with strong skills in:
Python, TensorFlow, PyTorch, Scikit-learn, Deep Learning, NLP,
Docker, Kubernetes, AWS, SQL, Git, Linux, Statistics, Spark.
Experience with data pipelines and model deployment is a plus.
"""
# ────────────────────────────────────────────────────────────

resume_text = parse_resume(RESUME_FILE)
result = skill_gap_analysis(resume_text, JOB_DESCRIPTION, CANDIDATE_NAME)
print_gap_report(result)


  SKILL GAP REPORT -- Anushka
  Match Score : 27%  (4/15 required skills)

  MATCHED (4)
    [+] Git
    [+] Machine Learning
    [+] Python
    [+] Scikit-Learn

  MISSING (11)
    [-] Aws                    [CORE    ] Cloud
    [-] Deep Learning          [ADVANCED] ML/AI
    [-] Docker                 [CORE    ] DevOps
    [-] Kubernetes             [ADVANCED] DevOps
    [-] Linux                  [CORE    ] DevOps
    [-] Nlp                    [ADVANCED] ML/AI
    [-] Pytorch                [ADVANCED] ML/AI
    [-] Spark                  [ADVANCED] Data
    [-] Sql                    [CORE    ] Databases
    [-] Statistics             [CORE    ] ML/AI
    [-] Tensorflow             [ADVANCED] ML/AI

  BONUS SKILLS (9)
    [*] Fastapi
    [*] Flask
    [*] Java
    [*] Javascript
    [*] Node.Js
    [*] Postgresql
    [*] React
    [*] Redis
    [*] Vue

  LEARNING ROADMAP (in order)
  -------------------------------------------------------

  [CORE SKILLS (Start here)]

    Aws (C

## Generate Visual HTML Report

In [ ]:
def generate_html_report(result, output_file="skill_gap_report.html"):
    pct = result['match_percent']
    ring_color = "#22c55e" if pct >= 75 else "#f59e0b" if pct >= 50 else "#ef4444"

    cat_colors = {
        "Programming": "#6366f1", "ML/AI": "#8b5cf6", "Web Dev": "#06b6d4",
        "Databases": "#f59e0b", "DevOps": "#10b981", "Cloud": "#3b82f6", "Data": "#ec4899",
    }

    def cat_badge(cat):
        c = cat_colors.get(cat, "#94a3b8")
        return f'<span style="background:{c}22;color:{c};border:1px solid {c}44;padding:2px 8px;border-radius:20px;font-size:11px;font-weight:600">{cat}</span>'

    def skill_chip(skill, color, icon):
        return (f'<span style="display:inline-flex;align-items:center;gap:4px;' +
                f'background:{color}18;color:{color};border:1px solid {color}33;' +
                f'padding:4px 10px;border-radius:20px;font-size:13px;margin:3px">' +
                f'{icon} {skill.title()}</span>')

    current_level = None
    missing_html = ""
    for rec in result['recommendations']:
        if rec['level'] != current_level:
            current_level = rec['level']
            if current_level == "core":
                missing_html += '<h3 style="color:#22d3ee;margin:24px 0 12px;font-size:14px;letter-spacing:2px;text-transform:uppercase">Core Skills — Learn First</h3>'
            else:
                missing_html += '<h3 style="color:#a78bfa;margin:32px 0 12px;font-size:14px;letter-spacing:2px;text-transform:uppercase">Advanced Skills — After Core</h3>'
        resources_html = "".join(
            f'<a href="{url}" target="_blank" style="display:flex;align-items:center;gap:6px;padding:6px 10px;background:#ffffff08;border-radius:6px;color:#cbd5e1;text-decoration:none;font-size:13px">&#8594; {name}</a>'
            for name, url in rec['resources']
        )
        missing_html += (
            f'<div style="background:#1e293b;border:1px solid #334155;border-radius:10px;padding:16px;margin-bottom:10px">' +
            f'<div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:10px">' +
            f'<span style="color:#f1f5f9;font-weight:600;font-size:15px">{rec["skill"].title()}</span>{cat_badge(rec["category"])}</div>' +
            f'<div style="display:flex;flex-direction:column;gap:4px">{resources_html}</div></div>'
        )

    matched_chips = " ".join(skill_chip(s, "#22c55e", "&#10003;") for s in result['matched']) or '<span style="color:#64748b">None matched</span>'
    extra_chips   = " ".join(skill_chip(s, "#f59e0b", "&#11088;") for s in result['extra_skills']) or '<span style="color:#64748b">None</span>'

    radius = 54
    circ   = 2 * 3.14159 * radius
    filled = circ * pct / 100
    donut = (
        f'<svg width="140" height="140" viewBox="0 0 140 140">' +
        f'<circle cx="70" cy="70" r="{radius}" fill="none" stroke="#1e293b" stroke-width="14"/>' +
        f'<circle cx="70" cy="70" r="{radius}" fill="none" stroke="{ring_color}" stroke-width="14" ' +
        f'stroke-dasharray="{filled:.1f} {circ:.1f}" stroke-dashoffset="{circ*0.25:.1f}" stroke-linecap="round"/>' +
        f'<text x="70" y="65" text-anchor="middle" fill="white" font-size="26" font-weight="bold" font-family="system-ui">{pct}%</text>' +
        f'<text x="70" y="85" text-anchor="middle" fill="#94a3b8" font-size="11" font-family="system-ui">match</text></svg>'
    )

    name = result["candidate"]
    n_missing = len(result["missing"])
    n_matched = len(result["matched"])
    n_required = len(result["jd_required"])

    html = f"""<!DOCTYPE html>
<html lang="en"><head><meta charset="UTF-8">
<title>Skill Gap Report - {name}</title>
<style>
* {{ box-sizing: border-box; margin: 0; padding: 0; }}
body {{ background: #0f172a; color: #e2e8f0; font-family: 'Segoe UI', system-ui, sans-serif; padding: 32px 16px; }}
.container {{ max-width: 860px; margin: 0 auto; }}
.header {{ background: linear-gradient(135deg, #1e293b 0%, #0f172a 100%); border: 1px solid #334155; border-radius: 16px; padding: 32px; margin-bottom: 24px; display: flex; justify-content: space-between; align-items: center; gap: 24px; flex-wrap: wrap; }}
.section {{ background: #1e293b; border: 1px solid #334155; border-radius: 12px; padding: 24px; margin-bottom: 20px; }}
.section-title {{ font-size: 13px; font-weight: 700; letter-spacing: 2px; text-transform: uppercase; color: #94a3b8; margin-bottom: 16px; }}
</style></head>
<body><div class="container">
<div class="header">
  <div>
    <div style="font-size:13px;color:#64748b;letter-spacing:2px;text-transform:uppercase;margin-bottom:6px">Skill Gap Analysis</div>
    <h1 style="font-size:28px;color:#f8fafc;margin-bottom:4px">{name}</h1>
    <div style="color:#64748b;font-size:14px">{n_matched} of {n_required} required skills matched &nbsp;&middot;&nbsp; {n_missing} gaps identified</div>
  </div>
  {donut}
</div>
<div class="section"><div class="section-title">Matched Skills</div><div>{matched_chips}</div></div>
<div class="section"><div class="section-title">Bonus Skills (You Have Extra)</div><div>{extra_chips}</div></div>
<div class="section">
  <div class="section-title">Learning Roadmap - {n_missing} Skills to Acquire</div>
  {missing_html if missing_html else '<p style="color:#64748b">No skill gaps!</p>'}
</div>
</div></body></html>"""

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"HTML report saved -> {output_file}")

generate_html_report(result, "skill_gap_report.html")


HTML report saved -> skill_gap_report.html


## Plug Into Full ATS Pipeline

In [ ]:
# ─── PLUG INTO YOUR FULL ATS PIPELINE ───────────────────────
# Call this instead of running modules separately.

def full_ats_pipeline(resume_file, jd_text, candidate_name="Candidate"):
    raw_text = parse_resume(resume_file)

    print("\n" + "="*60)
    print(f"  ATS PIPELINE -- {candidate_name}")
    print("="*60)

    # MODULE 1: Fraud Detection
    # Uncomment and paste your trained model + functions here:
    # skills   = extract_skills(raw_text)
    # features = extract_features(raw_text, skills)
    # fs       = scaler.transform([features])
    # pred     = iso.predict(fs)[0]
    # score    = iso.decision_function(fs)[0]
    # flags    = hard_flag_check(raw_text, features)
    # verdict  = interpret_result(pred, score, flags, raw_text)
    # print(f"  Fraud Check : {verdict}")

    # MODULE 2: Skill Gap Analysis
    gap = skill_gap_analysis(raw_text, jd_text, candidate_name)
    print(f"  Skill Match : {gap['match_percent']}%  ({len(gap['matched'])}/{len(gap['jd_required'])} skills)")
    print(f"  Gaps        : {', '.join(s.title() for s in gap['missing']) or 'None'}")
    print(f"  Bonus       : {', '.join(s.title() for s in gap['extra_skills']) or 'None'}")

    fname = candidate_name.replace(' ', '_')
    generate_html_report(gap, f"report_{fname}.html")
    return gap

# Example:
# result = full_ats_pipeline("resume.pdf", JOB_DESCRIPTION, "Riya Suryawanshi")